# GreenGate: a live demonstration

**A confidence-aware cascading framework for optimizing energy and cost in large language and multimodal model inference**

P. T. R. Hemachandra · 28998 · BSc (Hons) Computer Science · NSBM Green University

`pip install greengate` · [github.com/Rasath16/GreenGate](https://github.com/Rasath16/GreenGate) · MIT licence

---

### The one idea

A cascade asks a cheap model first and escalates to an expensive one only when the cheap model is unsure.
**The cheap model runs on every single query.** So each query that escalates is paid for *twice*: once for
the answer that was thrown away, and once for the answer that was kept.

No published cascading system counts that second payment. This library does, and counting it changes the
answer.

---

## How this notebook is organised

| | | |
|---|---|---|
| **PART A** | Preparation | Run on arrival. Downloads weights, loads models, runs both audits. 15 to 20 minutes. |
| **PART B** | The demonstration | Run in front of the panel. Every cell finishes in seconds. |

Part A leaves the models loaded in memory and the audit results in variables. **Do not restart the kernel
between the two parts.** Part B then routes real questions through models that are already warm, so the
inference is genuinely live without anybody waiting for a download.

**Settings → Accelerator → GPU T4 x2** and **Internet → On** before running anything.

---
---
# PART A · Preparation

### Run this before the viva. Nothing here is presented.

---

In [ ]:
# A1  Install the library from the Python Package Index.
#     This is the real published package, not a local copy. The [gpu] extra
#     adds pynvml, for reading the board power sensor, and bitsandbytes, so
#     the 7B large tiers can be loaded in 4-bit.
!pip install -q "greengate[gpu]"

import greengate
print('greengate version :', greengate.__version__)
print('public API        :', greengate.__all__)

In [ ]:
# A2  Confirm we have a real GPU. The profiler reads the board's own power
#     sensor through NVML; with no GPU it falls back to an estimate and labels it.
import torch, subprocess
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device         :', torch.cuda.get_device_name(0))
    print(subprocess.run(['nvidia-smi','--query-gpu=name,power.draw,power.limit,memory.total',
                          '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# A3  The two pairings we will compare, and the traffic we will measure them on.
#     The small tier is the one the thesis validated. Both large tiers are 7B
#     and both are quantised to 4-bit, so size is held constant between them.
SMALL      = 'Qwen/Qwen2.5-1.5B-Instruct'
LARGE_A    = 'mistralai/Mistral-7B-Instruct-v0.2'   # a different model family
LARGE_B    = 'Qwen/Qwen2.5-7B-Instruct'             # the same family as the small tier

# 24 questions of the kind a developer's application actually receives.
SAMPLE = [
    # short factual
    'What is the capital of France?',
    'Name three primary colours.',
    'What does HTTP status code 404 mean?',
    'Convert 45 degrees Celsius to Fahrenheit.',
    'Who wrote the play Hamlet?',
    'What does the acronym API stand for?',
    'How many minutes are there in a full day?',
    'What is the chemical symbol for gold?',
    # everyday technical
    'Write a SQL query to find the second highest salary in a table.',
    'Explain the difference between a process and a thread.',
    'What is the time complexity of binary search, and why?',
    'Explain what a database index does and when it helps.',
    'Write a Python function that reverses a linked list.',
    'What is the difference between TCP and UDP?',
    'Explain what a race condition is, with an example.',
    'How does HTTPS protect data while it is in transit?',
    # open-ended design
    'Explain why transformer attention is quadratic in sequence length, and what has been proposed to reduce it.',
    'Design a caching strategy for a news site whose traffic spikes unpredictably, and justify the eviction policy.',
    'Design a fault-tolerant message queue and justify the delivery guarantees you choose.',
    'How would you decide whether to shard a database, and what breaks when you do?',
    'Compare optimistic and pessimistic concurrency control, and say when each is appropriate.',
    'Design a rate limiter for a public API and justify the algorithm you choose.',
    'Explain the trade-offs between microservices and a monolith for a team of five engineers.',
    'How would you migrate a live production database with no downtime?',
    # open judgement, where a small model is most likely to struggle
    'Should a startup build its own authentication or buy it, and what changes that answer?',
    'When is eventual consistency unacceptable, and how would you detect that in advance?',
    'Design an observability strategy for a system you did not build and cannot change.',
    'How would you evaluate whether a machine learning model is safe to deploy?',
    'What is the right way to version a public API that already has users?',
    'Explain how you would debug a performance regression that only appears in production.',
    'How should a small team decide between Kubernetes and a managed platform?',
    'Design a schema migration process for a team that deploys several times a day.',
    'What are the trade-offs of event sourcing, and when would you avoid it?',
    'How would you reduce the carbon footprint of an existing inference service?',
    'Explain the risks of using a large language model to review code.',
    'How would you structure a retry policy that does not make an outage worse?',
    'What makes a good postmortem, and what makes a useless one?',
    'How would you decide what to log, given storage is not free?',
    'Design a feature flag system and say what it must never do.',
    'How would you test a distributed system for correctness under partition?',
]
print(len(SAMPLE), 'queries; escalation rates will come in steps of',
      f'{1/len(SAMPLE):.1%}')

# Both tiers are in the shipped calibration registry, so nothing has to be fitted.
import json, importlib.resources as res
presets = json.loads(res.files('greengate').joinpath('presets.json').read_text())
for m in (SMALL, LARGE_A, LARGE_B):
    p = presets.get(m)
    if p:
        print(f'{m:45s} T={p["temperature"]:<8} ECE {p["ece_before"]} -> {p["ece_after"]}')
    else:
        print(f'{m:45s} no preset; would self-calibrate with gw.calibrate()')

### A4 · Audit the *same-family* pairing first

We measure this pairing, keep its numbers, then release it. Two large models cannot stay resident at once
without one of them drawing idle power into the other's measurements, which is the attribution problem
again.

In [ ]:
import time, gc
t0 = time.time()

gw_b = greengate.GreenGate(small=SMALL, large=LARGE_B,
                           large_is_api=False, large_4bit=True, max_new_tokens=60)
_ = gw_b.route('Say hello.')          # warm up, so no measurement includes kernel compilation
gw_b._session = type(gw_b._session)()
print(f'same-family pairing loaded and warm in {time.time()-t0:.0f} s')

qwen_report = gw_b.audit(SAMPLE, verbose=False)
print('audit complete:', qwen_report['recommendation'])

del gw_b                               # release before loading the next large tier
gc.collect(); torch.cuda.empty_cache()
print('released; GPU memory in use:', f'{torch.cuda.memory_allocated()/1e9:.2f} GB')

### A5 · Load the cross-family pairing and audit it

This is the pairing that stays loaded for Part B, so the live routing uses it.

In [ ]:
t0 = time.time()
gw = greengate.GreenGate(small=SMALL, large=LARGE_A,
                         large_is_api=False, large_4bit=True, max_new_tokens=60)
_ = gw.route('Say hello.')
gw._session = type(gw._session)()
print(f'cross-family pairing loaded and warm in {time.time()-t0:.0f} s')

mistral_report = gw.audit(SAMPLE, verbose=False)
print('audit complete:', mistral_report['recommendation'])

### A6 · Probe the confidence signal

Measured once here so Part B can display it instantly.

In [ ]:
PROBE = [
    'What is the capital of France?',
    'Name three primary colours.',
    'What does HTTP status code 404 mean?',
    'Explain why transformer attention is quadratic in sequence length.',
    'Design a caching strategy for a news site with unpredictable spikes.',
]

tok, mod = gw._small.tokenizer, gw._small.model

def mean_entropy(q, T, n=25):
    text = tok.apply_chat_template([{'role':'user','content':q}],
                                   add_generation_prompt=True, tokenize=False)
    ids = tok(text, return_tensors='pt')['input_ids'].to(mod.device)
    es = []
    with torch.no_grad():
        for _ in range(n):
            lg = mod(ids).logits[0, -1].float()
            pr = torch.softmax(lg / T, -1)
            es.append(float(-(pr * torch.log2(pr.clamp_min(1e-12))).sum()))
            nxt = lg.argmax().view(1, 1)
            ids = torch.cat([ids, nxt], 1)
            if nxt.item() == tok.eos_token_id:
                break
    return sum(es) / len(es)

signal_probe = [(q, mean_entropy(q, 1.0), mean_entropy(q, gw.temperature)) for q in PROBE]
print('signal probed on', len(signal_probe), 'questions')

In [ ]:
# A7  Choose an operating point, then route the whole sample at it.
#
#     Break-even is a CEILING, not a target. On carbon alone the cheapest
#     setting is always to escalate nothing. Escalating buys quality, which
#     this library deliberately does not measure, so the rate is a judgement
#     the developer makes against their own quality floor. Here we take half
#     the break-even rate: a rule that can be stated out loud, and that keeps
#     a wide margin rather than creeping toward the line.
TARGET = mistral_report['break_even_escalation_rate'] / 2
row30  = min(mistral_report['rows'], key=lambda r: abs(r['realised_rate'] - TARGET))
gw.config(threshold=row30['threshold'])
gw._session = type(gw._session)()

# Route the whole sample at that setting. This gives Part B a real session,
# and tells us exactly which questions escalate, so the live demonstration
# can show one from each side of the gate rather than hoping.
routed    = [(q, gw.route(q)) for q in SAMPLE]
escalated = [(q, r) for q, r in routed if r.decision.startswith('ESCALATE')]
local     = [(q, r) for q, r in routed if r.decision == 'LOCAL']

if not escalated:            # insurance: never let the demonstration crash
    routed.sort(key=lambda x: x[1].signal)
    gw.config(threshold=routed[-2][1].signal - 1e-9)
    routed    = [(q, gw.route(q)) for q in SAMPLE]
    escalated = [(q, r) for q, r in routed if r.decision.startswith('ESCALATE')]
    local     = [(q, r) for q, r in routed if r.decision == 'LOCAL']
    print('no query escalated at the chosen point; gate lowered so both sides are visible')

DEMO_LOCAL     = min(local,     key=lambda x: x[1].signal)[0]   # most confident
DEMO_ESCALATED = max(escalated, key=lambda x: x[1].signal)[0]   # least confident

pre = gw.profile()
print('PART A COMPLETE')
print(f'  cost ratio measured     {mistral_report["cost_ratio_small_over_large"]}')
print(f'  break-even ceiling      {mistral_report["break_even_escalation_rate"]:.1%}')
print(f'  operating point chosen  {row30["realised_rate"]:.1%}  (half of break-even)')
print(f'  gate threshold          {row30["threshold"]:.4f}')
print()
print(f'  pre-run session         {pre["queries"]} queries, {pre["escalation_rate"]:.0%} escalated')
print(f'  questions that escalate {len(escalated)} of {len(SAMPLE)}')
print()
print('  chosen for the live demonstration:')
print(f'    stays local  ->  {DEMO_LOCAL[:66]}')
print(f'    escalates    ->  {DEMO_ESCALATED[:66]}')
print()
print('Leave this session open. Do not restart the kernel before Part B.')

---
---
# PART B · The demonstration

### Everything below runs in seconds. The routing executes live.

---
## 1 · The instrument

Before any routing, the library is a **measuring device**. `measure()` wraps *any* code, including
inference this library does not itself run, and reports what it cost.

> **Think of it like this.** A fuel gauge you can fit to a car somebody else built.

In [ ]:
with greengate.measure(n_queries=1, label='someone else\'s pipeline') as m:
    out = gw._small.generate('Explain what a cache miss is, in two sentences.')

print(m.report())
print()
print('energy, joules      :', round(m.energy_joules, 3))
print('carbon, grams CO2   :', round(m.carbon_grams, 8))
print('average power, watts:', round(m.avg_power_w, 1))
print('measurement mode    :', m.mode)

`mode` is the honesty field. On this GPU it reads **`gpu (pynvml)`**: the board's own power sensor, polled
every 100 ms and integrated over the run. On a laptop with no GPU it reads `cpu (estimated)` and the
library does not pretend otherwise.

Repeatability was measured at **1.4%** on text and **0.5%** on vision by re-running identical workloads.

---
## 2 · The instrument refuses to mis-measure

Device power cannot be split between two processes sharing a GPU. During the vision experiments one run
was launched twice by mistake; whole-device power attributed each run's consumption partly to the other and
**one tier came out 42% too high**. It was caught because that tier is measured in both runs and the values
disagreed.

Every measurement now records whether it had the accelerator to itself.

In [ ]:
with greengate.measure(n_queries=1, label='exclusivity check') as m:
    _ = gw._small.generate('What is 17 times 23?')

print('had the GPU to itself  :', m.exclusive)
print('other compute processes:', m.shared_with or 'none')
print()
print('With strict=True the measurement raises ContendedMeasurement instead of')
print('quietly returning a number that cannot be attributed to this process.')

---
## 3 · The accounting rule

A cascade runs the small tier on **every** query and the large tier on the escalated fraction. So against
simply using the large model for everything:

$$\text{saving} \;=\; 1 \;-\; \frac{C_{\text{small}}}{C_{\text{large}}} \;-\; e$$

where $e$ is the escalation rate. The cascade only saves while $e$ stays below $1 - C_s/C_l$.

> **Think of it like this.** If the small model is 47% of the price, that leaves 53% of headroom. Every
> query you escalate eats one point of it.

In [ ]:
from greengate import should_cascade

# Small model costs 47, large costs 100. Four kinds of traffic.
for label, rate in [('mostly easy questions', 0.10),
                    ('mixed traffic',          0.30),
                    ('half are hard',          0.53),
                    ('ALL questions are hard', 1.00)]:
    print(f'{label:24s} escalate {rate:4.0%}  ->  {should_cascade(47, 100, rate)["verdict"]}')

Read the last line. **If every query is hard, cascading is the worst thing you can do**: you pay for the
small model on every query and throw every one of those answers away.

That is the finding no previous cascading paper can express, because none of them charges for the
discarded run.

---
## 4 · The signal, and why it is calibrated

A model's raw confidence is overstated: it will claim 95% and be right 70% of the time. Calibration divides
the scores by one fitted constant so the claim matches reality. Here is what that does to the same
questions.

In [ ]:
print(f'calibration constant fitted for this model: T = {gw.temperature}')
print()
print(f"{'question':52s}{'raw':>9s}{'calibrated':>13s}")
for q, r, c in signal_probe:
    print(f'{q[:50]:52s}{r:9.3f}{c:13.3f}')

### Does the correction actually help?

Spacing is not the same thing as usefulness, and only a labelled test separates them. Measured on the 500
judged conversations in the thesis, with this same small model, where the label is whether the large tier
*actually* answered better:

| signal | AUROC | interval |
|---|---|---|
| raw entropy | 0.533 | 0.476 to 0.589, **includes chance** |
| calibrated entropy | **0.595** | 0.534 to 0.647, **excludes chance** |
| a classifier trained on the query text | 0.584 | needs labelled data for every model pair |

Calibration does what it is supposed to do: it moves the signal from indistinguishable from chance to
distinguishable, and it matches a trained classifier **while needing no training data at all**. It costs one
constant, fitted once, and it ships inside the library.

It is still a weak signal on open-ended conversation, and the thesis says so and bounds it: any routing
gain is at most 0.65 points of answer quality. **The accounting that follows does not depend on the signal
at all.** That is the contribution.

---
## 5 · Decide before deploying anything

A developer should not have to deploy a cascade to find out whether it helps. `audit()` runs **both** tiers
over a sample of their own queries, so the cost ratio is *measured* rather than assumed, and reports the
threshold that achieves each escalation rate together with the saving it would produce.

This was measured in Part A on the 24 questions above. Every row is a setting the developer could choose.

In [ ]:
r = mistral_report
print(f'audited {r["queries_audited"]} queries on this traffic')
print(f'  one small call costs   {r["mean_small_carbon_g"]:.6f} g')
print(f'  one large call costs   {r["mean_large_carbon_g"]:.6f} g')
print(f'  cost ratio             {r["cost_ratio_small_over_large"]:.4f}')
print(f'  break-even escalation  {r["break_even_escalation_rate"]:.1%}')
print()
print(f"  {'escalate':>9s}{'threshold':>12s}{'saving':>10s}")
for row in r['rows']:
    t = 'never' if row['threshold'] is None else f"{row['threshold']:.3f}"
    print(f"  {row['realised_rate']:>8.0%}{t:>12s}{row['predicted_saving']:>9.1%}")
print()
print(' ', r['recommendation'])

The `recommendation` line is the product. It does **not** promise a saving. It tells the developer whether
they are getting one, including when the honest answer is no.

`audit()` reports **cost, not quality**. The library contains no judge, and says so rather than inventing a
quality number.

---
## 6 · Routing, live

The gate is set at **half the break-even rate**. That is a deliberate choice, not an optimum: on carbon
alone escalating nothing is always cheapest, and the audit says so outright. Escalating buys quality, and
quality is the one thing this library does not measure, so the rate is a judgement a developer makes
against their own quality floor. Half of break-even keeps a wide margin of safety.

The two questions below were picked in Part A as the most and least confident in the audited sample, so
both sides of the gate are visible. They are routed **now**. Watch the `discarded` figure on the escalated
one.

In [ ]:
# Chosen in Part A from the audited sample: the most confident question and
# the least confident one, so both sides of the gate are visible.
QUESTIONS = [DEMO_LOCAL, DEMO_ESCALATED]

for q in QUESTIONS:
    res = gw.route(q)
    esc = res.decision.startswith('ESCALATE')
    print('=' * 92)
    print(q)
    print('-' * 92)
    print(f'  signal     {res.signal:.4f}      threshold {res.threshold:.4f}')
    print(f'  decision   {res.decision}')
    print(f'  answered by {res.large_model if esc else res.small_model}')
    print(f'  carbon     {res.carbon_g:.8f} g total')
    print(f'  discarded  {res.wasted_carbon_g:.8f} g   <-- the run that was thrown away')
    print(f'  latency    {res.latency_s:.2f} s')
    print()
    print('  ' + res.response.strip()[:280].replace('\n', '\n  '))
    print()

On an escalated query, `discarded` is non-zero. **That number is the contribution of this thesis.** Every
other cascading tool reports the query as costing only the large-model run.

The signal is weak on open-ended text, so on any individual pair of questions the gate can still choose
oddly. That is a measured property of the result, bounded in the thesis at 0.65 points of quality, not a
defect. What is never arbitrary is the accounting.

---
## 7 · The session verdict

`profile()` reports what this session actually emitted, applies the break-even condition to the per-tier
costs it measured, and returns a verdict in plain words.

In [ ]:
for k, v in gw.profile().items():
    print(f'  {k:32s} {v}')

The session behind this includes the sample routed in Part A plus the queries just run live, so there is
enough traffic for a verdict.

Had there not been, `verdict` would say **what is still missing** rather than guessing, for example *"no
verdict yet: needs 4 more escalations to measure the large tier"*. An instrument that does not know should
say so, and this one does.

---
## 8 · The carbon ceiling

A developer can cap emissions over a rolling window. When escalating would breach the cap, the router
returns the small-tier answer and records the degradation, rather than silently overrunning the budget or
failing the request.

In [ ]:
_ = gw.config(threshold=-1.0, budget_g=0.0005)   # force escalation, then refuse it with a tiny ceiling

for q in ['Explain what a deadlock is.',
          'Design a rate limiter for a public API.']:
    res = gw.route(q)
    print(f'{res.decision:16s}  signal {res.signal:.3f}   carbon {res.carbon_g:.8f} g')

print()
print('queries blocked by the carbon ceiling :', gw.profile()['budget_blocked'])
print()
print('LOCAL(budget) means the gate wanted to escalate and the ceiling refused.')
print('The answer still comes back. Quality degrades gracefully instead of the')
print('service failing or the budget being silently exceeded.')

_ = gw.config(threshold=row30['threshold'])      # restore the audited operating point

---
## 9 · The same small model, a different partner

Everything above used Qwen2.5-1.5B against **Mistral-7B**. Part A also measured it against **Qwen2.5-7B**,
from the same family as the small tier. Same parameter count, same quantisation, same 24 questions.

If the decision depended only on model *size*, nothing would change.

In [ ]:
def at(rep, rate):
    rows = [x for x in rep['rows'] if abs(x['target_rate'] - rate) < 1e-9]
    return rows[0]['predicted_saving'] if rows else float('nan')

print(f"{'':34s}{'Mistral-7B':>14s}{'Qwen2.5-7B':>14s}")
print(f"{'':34s}{'(cross-family)':>14s}{'(same family)':>14s}")
print('-' * 62)
print(f"{'cost of one small call, g':34s}"
      f"{mistral_report['mean_small_carbon_g']:14.6f}{qwen_report['mean_small_carbon_g']:14.6f}")
print(f"{'cost of one large call, g':34s}"
      f"{mistral_report['mean_large_carbon_g']:14.6f}{qwen_report['mean_large_carbon_g']:14.6f}")
print(f"{'cost ratio, small over large':34s}"
      f"{mistral_report['cost_ratio_small_over_large']:14.4f}{qwen_report['cost_ratio_small_over_large']:14.4f}")
print(f"{'break-even escalation rate':34s}"
      f"{mistral_report['break_even_escalation_rate']:13.1%}{qwen_report['break_even_escalation_rate']:14.1%}")
print(f"{'saving if you escalate 30%':34s}"
      f"{at(mistral_report, 0.3):13.1%}{at(qwen_report, 0.3):14.1%}")
print()
print('cross-family :', mistral_report['recommendation'])
print()
print('same family  :', qwen_report['recommendation'])

The small tier is **identical** in both columns. The large tiers are the same parameter count, quantised the
same way, answering the same 24 questions. Whatever difference appears in the cost ratio moves the
break-even rate and the saving available at every escalation budget with it.

Two models of the same size are not interchangeable. Tokeniser, vocabulary size, attention layout,
generation length and how each one decides when to stop all feed into energy per query, and none of that is
visible from a parameter count.

**This is the argument for the whole library.** You cannot read the break-even point off a spec sheet, and
you cannot reason it out from model sizes. It has to be measured, on your own two models, with your own
traffic. That is what `audit()` is for, and it takes minutes before you commit to a design.

---
## 10 · Two things added after the evaluation

### Services that batch their requests

Under continuous batching a device-level meter cannot attribute energy to one request, but it measures a
*window of work* correctly. `ServiceLedger` records each window with the tier that served it, then derives
the per-query costs, the realised escalation rate and the same break-even verdict from the totals.

This is the only honest option when per-request attribution is impossible.

In [ ]:
from greengate import ServiceLedger

ledger = ServiceLedger()
ledger.add(tier='small', queries=200, carbon_grams=0.0120)
ledger.add(tier='small', queries=190, carbon_grams=0.0115)
ledger.add(tier='large', queries=60,  carbon_grams=0.0620)
ledger.add(tier='large', queries=58,  carbon_grams=0.0605)

for k, v in ledger.report().items():
    print(f'  {k:32s} {v}')

### Where in the world the machine is

Grid carbon intensity varies by more than **twenty times** between regions. The default is the world average
of 475 gCO2/kWh, which could misstate a developer's real figure by an order of magnitude.

In [ ]:
for region, gco2 in [('Norway', 30), ('France', 56), ('world average', 475), ('India', 713)]:
    kwh = 1000.0 / 3_600_000          # the same one kilojoule of inference, everywhere
    print(f'  {region:16s} {gco2:>4} gCO2/kWh  ->  {kwh * 1.2 * gco2:.6f} g for the same work')

print()
print('Set it with:  greengate.GreenGate(small=..., large=..., carbon_intensity=56)')

---
## What this demonstration showed

| | |
|---|---|
| **1** | The library measures inference it does not run, from the GPU's own power sensor |
| **2** | It refuses to report a figure it cannot attribute, after that failure mode was found in my own data |
| **3** | The break-even condition decides, in plain words, whether cascading can pay at all |
| **4** | Calibration moves the signal from chance to better than chance, with no training data |
| **5** | `audit()` answers the deployment question on real traffic *before* anything is deployed |
| **6** | Routing charges escalated queries for both runs, and names the discarded one |
| **7** | `profile()` returns the session verdict, or says what it still needs to know |
| **8** | A carbon ceiling degrades quality gracefully instead of overrunning |
| **9** | Two large models of the same size give different break-even points, so it has to be measured |
| **10** | Batched services and regional grids are handled honestly rather than ignored |

---

**It does not promise a saving. It tells you whether you are getting one, including when the honest answer
is no.** That is the difference between a claim and an instrument.

---

*Every figure in the thesis can be recomputed from the public repository with no GPU and no API key:*

```
git clone https://github.com/Rasath16/GreenGate
cd GreenGate && python bootstrap_results.py && python eval_crossfamily.py
```